# Practice 2 · Causal Attention by Hand

| Marker | Meaning |
|---|---|
| 🟢 EDIT | you write this part |
| 🟡 READ | read it, no need to change |
| ⚪ RUN | just run it as it is |

### I. Background

Scaled dot-product attention turns queries and keys into attention weights, then uses those
weights to combine values. A causal mask prevents a position from attending to future tokens.
This practice builds the three stages separately so that every tensor can be checked.

### II. Task Requirements

1. **Q1** — write *scaled_scores(Q, K)* and *attention_weights(Q, K)*.
2. **Q2** — write *apply_causal_mask(scores)*.
3. **Q3** — write *causal_attention(Q, K, V)* and save the attention heatmap.

Steps: open the notebook in Haima, write the 🟢 parts, and run the cells from top to bottom.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| *scaled_scores(Q, K)* | scaled dot products | *QK^T / sqrt(d_k)* |
| *attention_weights(Q, K)* | softmax over keys | every row sums to 1 |
| *apply_causal_mask(scores)* | future positions masked | upper triangle is *-inf* |
| *causal_attention(Q, K, V)* | context and attention weights | shapes *(L, d_v)* and *(L, L)* |
| *attention_&lt;SID&gt;.png* | causal attention heatmap | one file in the working directory |

### IV. Who does what

- **Q1**: required for undergraduate students; postgraduates do it as well.
- **Q2–Q3**: required for postgraduate students; optional for undergraduates.
  An undergraduate who finishes advanced questions gets bonus marks; leaving them unfinished costs nothing.


## Question 1 · Scaled scores and attention weights

Write **one line in each function** to compute scaled scores and turn them into attention
weights. Support the unbatched shapes *(L, d_k)* and *(S, d_k)*. Do not apply a causal mask
in this question.

$$
S=\frac{QK^T}{\sqrt{d_k}}, \qquad A=\operatorname{softmax}(S)
$$

The matrix multiplication produces one score for every query–key pair. Softmax is applied
over the **last dimension**, so every query receives a probability distribution over keys.

**You may find these functions useful:**

- *Q @ K.transpose(-2, -1)*: matrix multiplication with the last two dimensions transposed.
- *math.sqrt(Q.size(-1))*: compute *sqrt(d_k)*.
- *torch.softmax(scores, dim=-1)*: normalize scores over keys.


In [ ]:
# 🟢 EDIT: set SID to the last two digits of your student ID.
SID = 23

import math
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(0)


def scaled_scores(Q, K):
    """Return QK^T / sqrt(d_k)."""
    # 🟢 CODE HERE: write 1 line with @, K.transpose(-2, -1), and math.sqrt(...).
    return Q @ K.transpose(-2,-1) /math.sqrt(Q.size(-1))


def attention_weights(Q, K):
    """Return softmax attention weights over the key dimension."""
    # 🟢 CODE HERE: write 1 line; reuse scaled_scores and set dim=-1.
    return torch.softmax(scaled_scores(Q , K) ,dim = -1 )


# ⚪ RUN: fixed small tensors make the results reproducible.
Q = torch.randn(4, 3)
K = torch.randn(4, 3)
V = torch.randn(4, 2)
weights = attention_weights(Q, K)
print("scores:\n", scaled_scores(Q, K))
print("row sums:", weights.sum(dim=-1))


scores:
 tensor([[ 0.3988, -0.2642, -1.0039,  0.1619],
        [-0.1053,  0.0936,  0.3531, -0.7361],
        [ 0.7778, -0.4004, -1.0917,  0.4721],
        [-0.2922,  0.2057,  0.7665, -0.5815]])
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000])


## Question 2 · Mask future positions

Write **3–5 lines** in *apply_causal_mask(scores)* for scores with shape *(L, S)* or
*(batch, L, S)*. This function prevents every query from reading future keys. Position *i*
may attend to key positions *j <= i*. The input tensor must remain unchanged.

$$
\widetilde{S}_{ij}=\begin{cases}
S_{ij}, & j\leq i\\
-\infty, & j>i
\end{cases}
$$

After masking, softmax gives exactly zero weight to every position where *j > i*.

**You may find these functions useful:**

- *scores.shape[-2:]*: read *L* and *S* for unbatched or batched scores.
- *torch.ones(L, S, dtype=torch.bool, device=scores.device)*: create a Boolean mask.
- *torch.triu(mask, diagonal=1)*: select positions above the main diagonal.
- *scores.masked_fill(mask, float("-inf"))*: replace future scores without modifying the input.


In [5]:
# 🟢 EDIT: write 3–5 lines.
def apply_causal_mask(scores):
    """Mask key positions after each query position."""
    #   1) read L and S from the last two dimensions
    #   2) create an upper-triangular Boolean mask with diagonal=1
    #   3) return scores.masked_fill(mask, float("-inf"))
    # YOUR CODE HERE
    L,S = scores.shape[-2:]
    mask = torch.triu(torch.ones((L, S), dtype=torch.bool, device=scores.device), diagonal=1)
    return scores.masked_fill(mask,float("-inf"))


# ⚪ RUN: recompute the weights after masking.
masked_scores = apply_causal_mask(scaled_scores(Q, K))
causal_weights = torch.softmax(masked_scores, dim=-1)
print("causal weights:\n", causal_weights)
print("future weights:", torch.triu(causal_weights, diagonal=1).sum().item())


causal weights:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4504, 0.5496, 0.0000, 0.0000],
        [0.6840, 0.2105, 0.1055, 0.0000],
        [0.1593, 0.2621, 0.4593, 0.1193]])
future weights: 0.0


## Question 3 · Combine values and inspect the pattern

Write **2–3 lines** in *causal_attention(Q, K, V)* using your functions from Q1 and Q2.
This function combines information from earlier positions and returns *(context, weights)*.
Then run the supplied plotting code and save the heatmap as *attention_&lt;SID&gt;.png*.

$$
\operatorname{CausalAttention}(Q,K,V)=
\operatorname{softmax}\left(\operatorname{CausalMask}
\left(\frac{QK^T}{\sqrt{d_k}}\right)\right)V
$$

**You may find these functions useful:** *scaled_scores(Q, K)*, *apply_causal_mask(scores)*,
*torch.softmax(..., dim=-1)*, and *weights @ V*. Reuse the earlier functions instead of
rewriting their internal steps.


In [7]:
# 🟢 EDIT: write 2–3 lines.
def causal_attention(Q, K, V):
    """Return (context, weights) for causal scaled dot-product attention."""
    #   1) compute weights with scaled_scores, apply_causal_mask, and softmax
    #   2) context = weights @ V
    #   3) return (context, weights)
    # YOUR CODE HERE
    weights = torch.softmax(apply_causal_mask(scaled_scores(Q,K)),dim = -1)
    context = weights @ V
    return context , weights

# ⚪ RUN: check shapes and save the attention heatmap.
context, weights = causal_attention(Q, K, V)
print("context shape:", tuple(context.shape))
print("weights shape:", tuple(weights.shape))

fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(weights.detach().cpu().numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(weights.shape[1]))
ax.set_yticks(range(weights.shape[0]))
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_title("Causal Attention")
fig.colorbar(im, ax=ax, label="Attention weight")
fig.tight_layout()
filename = f"attention_{SID}.png"
fig.savefig(filename, dpi=180)
plt.close(fig)
print("saved:", filename)


context shape: (4, 2)
weights shape: (4, 4)
saved: attention_23.png
